# Train crop ResNet-50 (frozen) + quality CSVs / train JSON

Best nail model so far: **binary ResNet-50, frozen backbone**, `unas_cropped`.

1. Resize each crop to 224, compute **picture** stats on that RGB (plus original w/h).
2. **Train CSV** — one row per **train** image (`image_id` + picture attributes).
3. Train the frozen head (same recipe as before).
4. **JSON** — from the train CSV only: p5, p50, p95, mean, std, n per attribute.
5. **Test CSV** — picture stats + **model** fields: `p`, entropy, distance to 0.45 / 0.50 / 0.60.

Picture stats use **deterministic** 224 resize (no flip/rotation) so the JSON is a stable train reference.

Upload `unas_cropped` to Drive. T4 GPU for training.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/unas_cropped"
OUT_DIR = "/content/drive/MyDrive/anemia/outputs"

In [ ]:
import copy
import csv
import json
import os
import re
from collections import Counter

import cv2
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

os.makedirs(OUT_DIR, exist_ok=True)

CLASS_NAMES_3 = ["Normal", "Leve", "Moderada"]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES_3)}
BIN_NAMES = ["Normal", "Anemia"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
RESIZE = 224
THRESHOLDS = (0.45, 0.50, 0.60)
PICTURE_ATTRS = [
    "brightness", "contrast", "saturation",
    "highlight_frac", "shadow_frac", "sharpness",
    "orig_w", "orig_h", "orig_area", "aspect_ratio",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
def parse_image_id(path):
    return os.path.splitext(os.path.basename(path))[0]

def load_rgb(path):
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def luma(rgb):
    r = rgb[:, :, 0].astype(np.float64)
    g = rgb[:, :, 1].astype(np.float64)
    b = rgb[:, :, 2].astype(np.float64)
    return 0.299 * r + 0.587 * g + 0.114 * b

def picture_stats_after_resize(rgb224):
    y = luma(rgb224)
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV)
    s = hsv[:, :, 1].astype(np.float64)
    v = hsv[:, :, 2].astype(np.float64)
    gray = cv2.cvtColor(rgb224, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return {
        "brightness": float(np.mean(y)),
        "contrast": float(np.std(y)),
        "saturation": float(np.mean(s)),
        "highlight_frac": float(np.mean(v > 250)),
        "shadow_frac": float(np.mean(y < 10)),
        "sharpness": float(np.var(lap)),
    }

def stats_for_path(path):
    rgb = load_rgb(path)
    h, w = rgb.shape[:2]
    rgb224 = cv2.resize(rgb, (RESIZE, RESIZE), interpolation=cv2.INTER_AREA)
    return {
        "image_id": parse_image_id(path),
        "filename": os.path.basename(path),
        "orig_w": int(w),
        "orig_h": int(h),
        "orig_area": int(w * h),
        "aspect_ratio": float(w / h),
        **picture_stats_after_resize(rgb224),
    }

def entropy_binary(p, eps=1e-12):
    p = float(np.clip(p, eps, 1.0 - eps))
    return float(-p * np.log(p) - (1.0 - p) * np.log(1.0 - p))

def write_csv(path, rows, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    print("wrote", path, "n=", len(rows))

In [ ]:
def list_image_folder_like(data_dir, class_to_idx):
    paths, labels = [], []
    for class_name in sorted(class_to_idx.keys()):
        folder = os.path.join(data_dir, class_name)
        for fname in sorted(os.listdir(folder)):
            if os.path.splitext(fname)[1].lower() not in IMAGE_EXTS:
                continue
            paths.append(os.path.join(folder, fname))
            labels.append(class_to_idx[class_name])
    return paths, np.array(labels, dtype=np.int64)

paths, y3 = list_image_folder_like(DATA_DIR, CLASS_TO_IDX)
y_bin = (y3 != 0).astype(np.int64)
class_names = [CLASS_NAMES_3[int(v)] for v in y3]
idx = list(range(len(paths)))

train_idx, test_idx = train_test_split(
    idx, test_size=0.20, stratify=y3, random_state=42
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, stratify=[y3[i] for i in train_idx], random_state=42
)
print("n", len(paths), "train", len(train_idx), "val", len(val_idx), "test", len(test_idx))
print("test binary", Counter(int(y_bin[i]) for i in test_idx))

## Picture stats on train (resize 224, no augment)

In [ ]:
train_rows = []
for k, i in enumerate(train_idx):
    row = stats_for_path(paths[i])
    row["class_name"] = class_names[i]
    row["y_bin"] = int(y_bin[i])
    row["split"] = "train"
    train_rows.append(row)
    if (k + 1) % 100 == 0 or k + 1 == len(train_idx):
        print(f"train stats {k + 1}/{len(train_idx)}")

train_csv_fields = [
    "image_id", "filename", "class_name", "y_bin", "split",
    "orig_w", "orig_h", "orig_area", "aspect_ratio",
    "brightness", "contrast", "saturation",
    "highlight_frac", "shadow_frac", "sharpness",
]
train_csv_path = os.path.join(OUT_DIR, "nail_quality_train.csv")
write_csv(train_csv_path, train_rows, train_csv_fields)

summary = {"n": len(train_rows), "resize": RESIZE, "split": {"random_state": 42}, "attributes": {}}
for name in PICTURE_ATTRS:
    vals = np.array([float(r[name]) for r in train_rows], dtype=np.float64)
    p5, p50, p95 = np.percentile(vals, [5, 50, 95])
    summary["attributes"][name] = {
        "p05": float(p5),
        "p50": float(p50),
        "p95": float(p95),
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals, ddof=0)),
        "n": int(len(train_rows)),
    }
json_path = os.path.join(OUT_DIR, "nail_quality_train_stats.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("wrote", json_path)
print(json.dumps(summary["attributes"]["brightness"], indent=2))

## Train frozen ResNet-50 (best model)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((RESIZE, RESIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((RESIZE, RESIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class NailBinaryDataset(Dataset):
    def __init__(self, paths, y, transform):
        self.paths = paths
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), int(self.y[i])

ds_train = NailBinaryDataset(paths, y_bin, train_tf)
ds_eval = NailBinaryDataset(paths, y_bin, eval_tf)
nw = 2 if device.type == "cuda" else 0
train_loader = DataLoader(Subset(ds_train, train_idx), batch_size=32, shuffle=True, num_workers=nw)
val_loader = DataLoader(Subset(ds_eval, val_idx), batch_size=32, shuffle=False, num_workers=nw)
test_loader = DataLoader(Subset(ds_eval, test_idx), batch_size=32, shuffle=False, num_workers=nw)

n_neg = int((y_bin[train_idx] == 0).sum())
n_pos = int((y_bin[train_idx] == 1).sum())
class_weights = torch.tensor(
    [len(train_idx) / (2 * n_neg), len(train_idx) / (2 * n_pos)],
    dtype=torch.float32,
    device=device,
)
print("weights", class_weights)

In [ ]:
def run_epoch(model, loader, criterion, optimizer, train):
    if train:
        model.eval()
        model.fc.train()
    else:
        model.eval()
    total_loss, n = 0.0, 0
    y_true, y_pred = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * y.size(0)
        n += y.size(0)
        y_true.extend(y.cpu().tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    return total_loss / n, acc, f1

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
for p in model.parameters():
    p.requires_grad = False
model.fc = torch.nn.Linear(model.fc.in_features, 2)
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

best_f1, best_state = -1.0, None
for epoch in range(1, 16):
    tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, opt, True)
    va_loss, va_acc, va_f1 = run_epoch(model, val_loader, criterion, None, False)
    print(
        f"frozen {epoch:02d}  train loss {tr_loss:.3f} acc {tr_acc:.3f} f1 {tr_f1:.3f}  "
        f"val loss {va_loss:.3f} acc {va_acc:.3f} f1 {va_f1:.3f}"
    )
    if va_f1 > best_f1:
        best_f1 = va_f1
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
ckpt = os.path.join(OUT_DIR, "resnet50_binary_crop_frozen_best.pt")
torch.save(best_state, ckpt)
print("best val anemia F1", best_f1, "saved", ckpt)

## Test CSV: picture stats + p, entropy, distance to thresholds

In [ ]:
model.eval()
test_rows = []
y_true_all, p_all = [], []

with torch.no_grad():
    for i in test_idx:
        row = stats_for_path(paths[i])
        tensor, y = ds_eval[i]
        logits = model(tensor.unsqueeze(0).to(device))
        p = float(torch.softmax(logits, dim=1)[0, 1].cpu())
        row["class_name"] = class_names[i]
        row["y_bin"] = int(y_bin[i])
        row["split"] = "test"
        row["p"] = p
        row["entropy"] = entropy_binary(p)
        for t in THRESHOLDS:
            tag = f"{float(t):.2f}".replace(".", "")
            row[f"dist_t{tag}"] = abs(p - t)
            row[f"pred_t{tag}"] = int(p >= t)
        test_rows.append(row)
        y_true_all.append(int(y_bin[i]))
        p_all.append(p)

test_csv_fields = train_csv_fields + [
    "p", "entropy",
    "dist_t045", "pred_t045",
    "dist_t050", "pred_t050",
    "dist_t060", "pred_t060",
]
test_csv_path = os.path.join(OUT_DIR, "nail_quality_test.csv")
write_csv(test_csv_path, test_rows, test_csv_fields)

y_true_all = np.array(y_true_all)
p_all = np.array(p_all)
print("test AUC", roc_auc_score(y_true_all, p_all))
for t in THRESHOLDS:
    pred = (p_all >= t).astype(int)
    print(
        f"t={t:.2f}  prec {precision_score(y_true_all, pred, zero_division=0):.3f}  "
        f"rec {recall_score(y_true_all, pred, zero_division=0):.3f}  "
        f"F1 {f1_score(y_true_all, pred, zero_division=0):.3f}"
    )
print("confusion t=0.50")
print(confusion_matrix(y_true_all, (p_all >= 0.50).astype(int)))